In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import time

In [2]:
# Set constants and paths
data_dir = "image-results-spoofing/Data"
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")
batch_size = 32
learning_rate = 0.001
num_epochs = 30
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()
MODEL_NAME = "CNN-LSTM"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../UAVATTACK_Results.csv"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 82)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 82)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

In [5]:
# Datasets and DataLoaders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'test': test_loader}
classes = train_dataset.classes

In [6]:
class CNN_LSTM_Model(nn.Module):
    def __init__(self, num_classes, hidden_dim=128, lstm_layers=1):
        super(CNN_LSTM_Model, self).__init__()
        self.layer1 = self.ConvModule(in_features=1, out_features=64)
        self.layer2 = self.ConvModule(in_features=64, out_features=128)
        self.layer3 = self.ConvModule(in_features=128, out_features=256)
        self.layer4 = self.ConvModule(in_features=256, out_features=512)
        self.gap = nn.AdaptiveAvgPool2d((2, 2))
        self.lstm_input_size = 512 * 2
        self.lstm = nn.LSTM(input_size=self.lstm_input_size, hidden_size=hidden_dim, num_layers=lstm_layers, batch_first=True, bidirectional=False)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)

        x = x.permute(0, 2, 1, 3)
        x = x.contiguous().view(batch_size, 2, -1)
        lstm_out, (h_n, c_n) = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.classifier(out)
        return out

    def ConvModule(self, in_features, out_features):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_features),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )


In [7]:
# Instantiate model, loss, optimizer
model = CNN_LSTM_Model(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [8]:
# Training loop
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
best_epoch = 0

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    for inputs, labels in dataloaders['train']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_train_preds.append(preds.cpu().numpy())
        all_train_labels.append(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_dataset)
    train_preds = np.concatenate(all_train_preds)
    train_labels = np.concatenate(all_train_labels)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation on test set
    model.eval()
    all_val_preds = []
    all_val_labels = []
    with torch.no_grad():
        for inputs, labels in dataloaders['test']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
    val_preds = np.concatenate(all_val_preds)
    val_labels = np.concatenate(all_val_labels)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

train_time = time.time() - start_time

model.load_state_dict(best_model_wts)

Epoch 1/30 - Train Loss: 0.6675 - Train Acc: 0.8034 - Val Acc: 0.8631
Epoch 2/30 - Train Loss: 0.4221 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 3/30 - Train Loss: 0.4096 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 4/30 - Train Loss: 0.4048 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 5/30 - Train Loss: 0.4025 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 6/30 - Train Loss: 0.4007 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 7/30 - Train Loss: 0.3966 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 8/30 - Train Loss: 0.3896 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 9/30 - Train Loss: 0.3825 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 10/30 - Train Loss: 0.3651 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 11/30 - Train Loss: 0.3430 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 12/30 - Train Loss: 0.3029 - Train Acc: 0.8648 - Val Acc: 0.8741
Epoch 13/30 - Train Loss: 0.2638 - Train Acc: 0.8962 - Val Acc: 0.9433
Epoch 14/30 - Train Loss: 0.2290 - Train Acc: 0.9193 - Val Acc: 0.2628
Epoch 15/30 - T

<All keys matched successfully>

In [9]:
# Final evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

Test Accuracy: 99.31%
Test F1 Score: 0.9931


In [12]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(model.state_dict().keys()),
    "num_epochs": num_epochs
}]
results_df = pd.DataFrame(results)
file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../UAVATTACK_Results.csv


In [11]:
# Save best model
model_save_path = "best_cnn_lstm_model.pth"
checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}
torch.save(checkpoint, model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to best_cnn_lstm_model.pth
